In [1]:
import torch

model_path = "../results/models/resnet50_eurosat.pth"
checkpoint = torch.load(model_path, map_location="cpu")

# See what's inside
if isinstance(checkpoint, dict):
    print("Keys:", list(checkpoint.keys()))
else:
    print("Type:", type(checkpoint))

Keys: ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.0.conv3.weight', 'layer1.0.bn3.weight', 'layer1.0.bn3.bias', 'layer1.0.bn3.running_mean', 'layer1.0.bn3.running_var', 'layer1.0.bn3.num_batches_tracked', 'layer1.0.downsample.0.weight', 'layer1.0.downsample.1.weight', 'layer1.0.downsample.1.bias', 'layer1.0.downsample.1.running_mean', 'layer1.0.downsample.1.running_var', 'layer1.0.downsample.1.num_batches_tracked', 'layer1.1.conv1.weight', 'layer1.1.bn1.weight', 'layer1.1.bn1.bias', 'layer1.1.bn1.running_mean', 'layer1.1.bn1.running_var', 'layer1.1.bn1.num_batches_tracked', 'layer1.1.conv2.weight'

In [2]:
print("FC layer shape:", checkpoint["fc.weight"].shape)
print("Input channels (conv1):", checkpoint["conv1.weight"].shape)

FC layer shape: torch.Size([10, 2048])
Input channels (conv1): torch.Size([64, 3, 7, 7])


In [3]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# Load AlbaniaSAT data — RGB only (first 3 bands)
X = np.load("../data/AlbaniaSAT/processed/patches.npy")
y = np.load("../data/AlbaniaSAT/processed/labels.npy")

X_rgb = X[:, :3, :, :]  # drop NIR, keep RGB only

# Normalize same way as ImageNet (what EuroSAT used)
mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = X_rgb / 10000.0  # Sentinel-2 values are 0-10000
X_rgb = (X_rgb - mean) / std

# Build dataloader
X_tensor = torch.tensor(X_rgb, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=False)

print(f"Dataset: {X_tensor.shape}")
print("Ready to evaluate!")

Dataset: torch.Size([4000, 3, 64, 64])
Ready to evaluate!


In [5]:
# Load EuroSAT model
model = models.resnet50()
model.fc = nn.Linear(2048, 10)  # 10 EuroSAT classes
model.load_state_dict(checkpoint)
model.eval()

# EuroSAT class names (original order)
eurosat_classes = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake"
]

class_names = [
    "Broad-leaved_Forest", "Coniferous_Forest", "Shrubland",
    "Agricultural", "Grassland", "Olive_Groves", "Urban", "Water"
]

# Run inference
all_preds = []
all_probs = []

with torch.no_grad():
    for batch_X, _ in loader:
        outputs = model(batch_X)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

albaniasat_classes = [c.replace("_", " ") for c in class_names]

print("EuroSAT model predictions on AlbaniaSAT data:\n")
print(f"{'AlbaniaSAT Class':<25} {'Top EuroSAT Prediction':<25} {'Confidence'}")
print("-" * 65)

for i, class_name in enumerate(albaniasat_classes):
    class_probs = all_probs[i * 500:(i + 1) * 500]
    avg_probs = class_probs.mean(axis=0)
    top_pred = eurosat_classes[np.argmax(avg_probs)]
    confidence = avg_probs.max()
    print(f"{class_name:<25} {top_pred:<25} {confidence:.2%}")

EuroSAT model predictions on AlbaniaSAT data:

AlbaniaSAT Class          Top EuroSAT Prediction    Confidence
-----------------------------------------------------------------
Broad-leaved Forest       SeaLake                   99.80%
Coniferous Forest         SeaLake                   99.72%
Shrubland                 SeaLake                   98.27%
Agricultural              SeaLake                   96.34%
Grassland                 SeaLake                   84.27%
Olive Groves              SeaLake                   99.65%
Urban                     SeaLake                   98.92%
Water                     SeaLake                   99.99%


In [6]:
X = np.load("../data/AlbaniaSAT/processed/patches.npy")
print("Min:", X.min())
print("Max:", X.max())
print("Mean:", X.mean())
print("Std:", X.std())

Min: 1.0
Max: 12880.0
Mean: 1278.5293
Std: 1149.8451


In [7]:
X_rgb = X[:, :3, :, :]

# Correct normalization for Sentinel-2
# Clip to valid range first, then normalize to 0-1
X_rgb = np.clip(X_rgb, 0, 3000) / 3000.0

# Then apply ImageNet normalization
mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)
X_rgb = (X_rgb - mean) / std

print("After normalization:")
print("Min:", X_rgb.min())
print("Max:", X_rgb.max())
print("Mean:", X_rgb.mean())

# Rebuild dataloader
X_tensor = torch.tensor(X_rgb, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=False)
print("Dataloader rebuilt!")

After normalization:
Min: -2.116448326070903
Max: 2.6399999999999997
Mean: -0.8776781791147984
Dataloader rebuilt!


In [8]:
all_preds = []
all_probs = []

with torch.no_grad():
    for batch_X, _ in loader:
        outputs = model(batch_X)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.numpy())
        all_probs.extend(probs.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

albaniasat_classes = [c.replace("_", " ") for c in class_names]

print("EuroSAT model predictions on AlbaniaSAT data:\n")
print(f"{'AlbaniaSAT Class':<25} {'Top EuroSAT Prediction':<25} {'Confidence'}")
print("-" * 65)

for i, class_name in enumerate(albaniasat_classes):
    class_probs = all_probs[i * 500:(i + 1) * 500]
    avg_probs = class_probs.mean(axis=0)
    top_pred = eurosat_classes[np.argmax(avg_probs)]
    confidence = avg_probs.max()
    print(f"{class_name:<25} {top_pred:<25} {confidence:.2%}")

EuroSAT model predictions on AlbaniaSAT data:

AlbaniaSAT Class          Top EuroSAT Prediction    Confidence
-----------------------------------------------------------------
Broad-leaved Forest       SeaLake                   77.60%
Coniferous Forest         SeaLake                   63.71%
Shrubland                 SeaLake                   57.00%
Agricultural              PermanentCrop             48.33%
Grassland                 PermanentCrop             53.65%
Olive Groves              PermanentCrop             49.21%
Urban                     PermanentCrop             85.22%
Water                     SeaLake                   93.49%


In [9]:
import pandas as pd

results = []
for i, class_name in enumerate(albaniasat_classes):
    class_probs = all_probs[i * 500:(i + 1) * 500]
    avg_probs = class_probs.mean(axis=0)
    top_pred = eurosat_classes[np.argmax(avg_probs)]
    confidence = avg_probs.max()
    results.append({"AlbaniaSAT Class": class_name, 
                    "Top EuroSAT Prediction": top_pred, 
                    "Confidence": f"{confidence:.2%}"})

df = pd.DataFrame(results)
df.to_csv("../results/eurosat_on_albaniasat.csv", index=False)
print(df.to_string(index=False))

   AlbaniaSAT Class Top EuroSAT Prediction Confidence
Broad-leaved Forest                SeaLake     77.60%
  Coniferous Forest                SeaLake     63.71%
          Shrubland                SeaLake     57.00%
       Agricultural          PermanentCrop     48.33%
          Grassland          PermanentCrop     53.65%
       Olive Groves          PermanentCrop     49.21%
              Urban          PermanentCrop     85.22%
              Water                SeaLake     93.49%
